> `static_retrieve_langchain.ipynb task `: 
> 完成`LangChain Retriever deterministic pipeline / LLM-adaptive pipeline`，把套`langchain langchain-core基础检索算法`汇总整合，然后逐个对应到 LangChain 官方类和底层算法。

```
查询规范化
→ 工程 metadata route
→ 中文词面/BM25 召回
→ Dense 补充召回
→ Weighted RRF
→ 完整任务记录定位
→ Evidence Gate
→ 引用回答或拒答
```

> LangChain Retriever: `langchain`,`langchain-core`,`langchain-classic`（高阶模块）

> 固定语料 V1 V2 V3 v4
    -> 确定性 Document / chunk
    -> `normalized embedding` + Chroma 
    -> metadata 工程路由
    -> dense + 中文 n-gram BM25
    -> weighted RRF
    -> top-k Document + 评分信号

> 召回和排序 --> 生成、LangGraph 或 Agent。`注意：Agent 决定"要不要检索"，并不会自动提高检索质量。`




| 包 | 主要职责 |
|---|---|
| `langchain-core` | `Document`、`BaseRetriever`、`VectorStore`、`VectorStoreRetriever`、Runnable 等基础接口 |
| `langchain` | 当前主包，主要承载 Agent、Middleware 等高层入口 |
| `langchain-community` | `BM25Retriever`、`TFIDFRetriever` 和社区数据源集成 |
| `langchain-classic` | `EnsembleRetriever`、`MultiQueryRetriever`、`MultiVectorRetriever` 等旧式检索编排组件 |
| `langchain-chroma` 等 | 具体 VectorStore 实现及其搜索能力 |
---
Retriever 本质上是“输入字符串，返回 Document 列表”的接口，比 VectorStore 更宽泛；VectorStore 可以转换成 Retriever。



| 路线 | 文档表示 | 排序依据 | LangChain 对应 |
|---|---|---|---|
| Dense Retrieval | Embedding 稠密向量 | Cosine、Inner Product、L2；底层可用 KNN/ANN | `VectorStoreRetriever` |
| Sparse Retrieval | Token、词频、倒排索引 | BM25、TF-IDF | `BM25Retriever`、`TFIDFRetriever` |
| Hybrid Retrieval | Dense + Sparse | RRF、Weighted RRF 或数据库原生融合 | `EnsembleRetriever` 或 VectorStore 原生 Hybrid |
| Reranking | Query-Document 对 | Cross-Encoder、专用 reranker | `ContextualCompressionRetriever` + compressor |



----
> 基础检索retrieve
- `search_type=similarity`：直接取最相关的 `k` 条，适合事实问答基线。`3种算法【Cosine、Inner Product、L2 Distance】`，一般是内置任意一种。Cosine --> 比较向量方向，忽略长度；越大越相似。Inner Product --> 同时受方向和向量模长影响；向量归一化后与 Cosine 排序等价。 L2 Distance --> 欧氏空间直线距离；越小越相似。 
- `search_type=mmr`：`mmr Maximal Marginal Relevance`--> 它也不是新的 Embedding 模型或 ANN 索引。即 Maximal Marginal Relevance，先进行向量搜索得到 fetch_k 个候选，再逐个选择兼顾相关性与多样性的文档。从更大的 `fetch_k` 候选池中平衡相关性与多样性，适合“概括多个方面”；它不保证精确事实问答更好。`fetch_k` 必须明显大于 `k` 才有意义。` k：最终返回数量。fetch_k：进入 MMR 重排的候选数量，应明显大于 k。lambda_mult=1：更重视查询相关性。lambda_mult=0：更重视结果多样性。`
- `similarity_score_threshold`：过滤低相关结果，适合无答案保护；阈值必须用当前 embedding + vector store 的评测分布校准。`similarity search 后加一道最低相关度门槛`，`score_threshold` 只属于 `similarity_score_threshold`。--> Top-K 强制返回 K 个文档，即使知识库里根本没有答案。


- `bm25`: BM25 是词面检索，不理解语义。 BM25 对“词”做匹配，但中文句子天然没有空格，所以必须先决定什么算一个词。这个过程就是分词（tokenization）。
```
Term Frequency：词在文档中出现越多，得分越高，但会饱和，不是无限线性增长。
IDF：越少见的词权重越高，因此工程名、设备名、任务名很有价值。
Length Normalization：对长文档做长度校正，避免长文档仅因包含更多词而占优。
缺点：无法自然理解同义改写，对分词非常敏感。
```

- `Hybrid Search：Dense+Sparse--Retriever--Ranking -->  BM25 Retriever--Ranking --> Weighted RRF --> final Ranking`: Weighted RRF(Reciprocal Rank Fusion) 它只依赖各检索器中的排名，不要求 BM25 score 与 cosine score 位于同一数值尺度。EnsembleRetriever 默认 c=60；相同文档通过 id_key 去重，没有 id_key 时默认使用 page_content。

```
Weighted RRF：对排名倒数加权。
Weighted Score Fusion：直接融合归一化后的原始分数。
原生 Hybrid：由 Pinecone、Elasticsearch、Weaviate 等后端自行融合，API 与公式不统一。
```

| 组件 | 实际作用 | 主要代价 |
|---|---|---|
| `RePhraseQueryRetriever` | LLM 改写一次查询后检索 | 增加模型调用，可能改错实体 |
| `MultiQueryRetriever` | LLM 生成多个查询，合并去重结果 | 提高召回，也增加延迟和噪声 |
| Metadata Filter | 在搜索前限定工程、版本、文档等 metadata | 依赖 metadata 质量和后端支持 |
| `SelfQueryRetriever` | LLM 将问题转成语义查询和结构化 metadata filter | 可解释性和稳定性低于确定性路由 |
| `MultiVectorRetriever` | 一个父文档对应多个可检索向量，命中子向量后返回父文档 | 需要维护 vectorstore 与 docstore |
| `ParentDocumentRetriever` | 用小 chunk 检索，返回较大的父级上下文 | 返回上下文可能变大 |
| `ContextualCompressionRetriever` | 对基础召回结果进行过滤、抽取或 rerank | 增加一次模型或 reranker 推理 |
------
官方定义中，MultiQueryRetriever 会生成多个查询并返回结果并集；ParentDocumentRetriever 是 MultiVectorRetriever 的具体父子文档方案；ContextualCompressionRetriever 包装基础 Retriever 和 compressor。


---
> `langchain-classic`: 承载 Ensemble、MultiQuery、MultiVector 等较高层检索策略,`retrieved orchestration / retrieval strategy`，`MultiVectorRetriever`高级组件。`Query Rewrite、Multi-Query、Metadata Filter、Parent / MultiVector、Contextual Compression、Self-Query`



```html
                 Embedding
Query ─────────────────────────→ Query Vector
                                      │
                                      ▼
                               Vector Store
                                      │
                    ┌─────────────────┼─────────────────┐
                    │                 │                 │
                    ▼                 ▼                 ▼
              Similarity        Score Threshold       MMR
                    │                 │                 │
                 Top-K          similarity > t    Top-N candidate
                                                       │
                                                       ▼
                                                   MMR rerank
                                                       │
                                                       ▼
                                                     Top-K

```

```html
                         Query
                           │
              ┌────────────┴─────────────┐
              │                          │
              ▼                          ▼
        Dense Retriever            BM25 Retriever
              │                          │
      Embedding Similarity          lexical match
              │                          │
           Top 20                     Top 20
              │                          │
              └────────────┬─────────────┘
                           ▼
                         Fusion
                           │
                           ▼
                          RRF
                           │
                           ▼
                         Top 20
                           │
                           ▼
                       Reranker
                           │
                           ▼
                          Top 5

```

```html
Retrieval
│
├── 1. Dense Retrieval
│      │
│      ├── Embedding
│      │
│      ├── Similarity
│      │      ├── Cosine
│      │      ├── Inner Product
│      │      └── L2
│      │
│      ├── Search Strategy
│      │      ├── Top-K Similarity
│      │      ├── Score Threshold
│      │      └── MMR
│      │
│      └── Vector Index
│             ├── Exact / Flat
│             ├── HNSW
│             ├── IVF
│             └── PQ / IVF-PQ
│
├── 2. Sparse / Lexical Retrieval
│      │
│      ├── TF-IDF
│      ├── BM25
│      └── Learned Sparse
│             └── SPLADE
│
├── 3. Hybrid Retrieval
│      │
│      ├── Dense
│      ├── Sparse
│      │
│      └── Fusion
│             ├── RRF
│             └── Weighted RRF
│
└── 4. Re-ranking
       │
       ├── Cross Encoder
       ├── BGE Reranker
       ├── Cohere Rerank
       └── LLM Rerank
```

```html
                     langchain-core
                           │
                     BaseRetriever
                           │
          ┌────────────────┴───────────────┐
          │                                │
 VectorStoreRetriever                其他 Retriever
          │
          │
   VectorStore Interface
          │
 ┌────────┼────────────┐
 │        │            │
similarity MMR      threshold
 │
 ▼
Vector DB
 │
 ├─ cosine / IP / L2
 │
 └─ Flat / HNSW / IVF / PQ
```

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path | None = None) -> Path:
    """找到仓库根目录，不依赖 Jupyter 从哪个目录启动。"""

    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        corpus = candidate / "knowledge" / "project_progress" / "texts"
        if corpus.is_dir() and (candidate / "ZZworkbench").is_dir():
            return candidate
    raise FileNotFoundError("Cannot locate the pipelines_rag repository root")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ZZworkbench.rag_langchain.text_retrieval import (
    DEFAULT_CORPUS_ROOT,
    DEFAULT_INDEX_ROOT,
    ChunkConfig,
    EmbeddingConfig,
    audit_text_corpus,
    build_embeddings,
    build_hybrid_retriever,
    build_or_reuse_chroma,
    chunk_statistics,
    evaluate_retriever,
    load_eval_cases,
    load_txt_documents,
    open_chroma,
    retrieve,
    retrieve_hybrid,
    split_documents,
)

print(f"repo: {REPO_ROOT}")


## 1. 先固定语料边界

线上基线只索引当前整理版本 `v4`。如果要比较 v1-v4，应分别建立四个 index，用同一问题集评测；不要把四版同时塞进一个 index。


In [ ]:
corpus_audit = audit_text_corpus(DEFAULT_CORPUS_ROOT)
print("versions:", corpus_audit["versions"])
print("exact duplicate groups:", len(corpus_audit["exact_duplicate_groups"]))

documents = load_txt_documents(DEFAULT_CORPUS_ROOT, version="v4")
chunk_config = ChunkConfig(chunk_size=960, chunk_overlap=160)
chunks = split_documents(documents, chunk_config)

print(f"documents={len(documents)}, chunks={len(chunks)}")
print(chunk_statistics(chunks))


In [ ]:
# 每个 chunk 都保留稳定 ID、父文档 ID、来源、版本和字符偏移。
sample = next(
    chunk
    for chunk in chunks
    if "江湾变电站及配套线路投运" in chunk.page_content
)
print("chunk id:", sample.id)
print("source:", sample.metadata["source"])
print("version:", sample.metadata["corpus_version"])
print("span:", sample.metadata["start_index"], sample.metadata["end_index"])
print(sample.page_content)


## 2. Index 是一份可校验的契约

索引不仅是一个 Chroma 目录，还必须记录语料哈希、chunk 配置、embedding 配置和 collection。配置完全一致时复用；不一致时显式重建，避免用另一种 embedding 查询旧向量。

这里使用明确的模型路径和 normalized embeddings，不再通过目录下标猜模型。


In [ ]:
embedding_config = EmbeddingConfig()
embeddings = build_embeddings(embedding_config)

index_result = build_or_reuse_chroma(
    documents,
    chunks,
    embeddings,
    embedding_config,
    persist_directory=DEFAULT_INDEX_ROOT,
    corpus_root=DEFAULT_CORPUS_ROOT,
    version="v4",
    chunk_config=chunk_config,
)
print(index_result)

# build_or_reuse_chroma 已校验 manifest；查询时打开同一个 collection。
vector_store = open_chroma(DEFAULT_INDEX_ROOT, embeddings)
print("stored chunks:", vector_store._collection.count())


In [ ]:
def compact(text: str, limit: int = 180) -> str:
    """压缩展示文本，但不隐藏来源。"""

    one_line = " ".join(text.split())
    return one_line if len(one_line) <= limit else one_line[:limit] + "..."


def show_hits(hits) -> None:
    """同时展示最终排名和 dense/BM25 的分路排名。"""

    for hit in hits:
        metadata = hit.document.metadata
        source = Path(str(metadata.get("source", ""))).name
        signals = []
        if metadata.get("dense_rank") is not None:
            signals.append(f"dense#{metadata['dense_rank']}")
        if metadata.get("lexical_rank") is not None:
            signals.append(f"bm25#{metadata['lexical_rank']}")
        detail = ", ".join(signals) or "dense"
        print(f"#{hit.rank} score={hit.score:.6f} [{detail}] {source}")
        print(compact(hit.document.page_content), "\n")


## 3. Dense retrieval 是基线，不是最终答案

Dense retrieval 擅长语义改写，但工程名、任务 ID、电压等级和日期通常更依赖精确词面匹配。先保留 dense 作为对照，才能知道 hybrid 是否真的带来提升。

注意：Chroma 的 `similarity_search_with_score()` 在当前 cosine collection 中返回的是 **distance（越小越好）**。仓库包装层同时暴露 `distance` 和 `1 - distance`，这个 score 便于排序，但不是概率，也不应跨模型直接比较。


In [ ]:
query = "珠海110kV江湾输变电工程的江湾变电站及配套线路投运任务什么时候完成？"
dense_hits = retrieve(vector_store, query, k=4)
show_hits(dense_hits)


## 4. 修正原来的 coroutine 错误

两条调用路径不要混用：

    # 传查询字符串：vector store 自己调用 embed_query
    pairs = vector_store.similarity_search_with_score(query, k=4)

    # 传已计算的向量：调用 by_vector 方法
    query_vector = embeddings.embed_query(query)
    docs = vector_store.similarity_search_by_vector(query_vector, k=4)

    # 真正的异步写法需要 await
    query_vector = await embeddings.aembed_query(query)
    docs = await retriever.ainvoke(query)

下面只执行同步版本，便于和其余实验保持一致。


In [ ]:
query_vector = embeddings.embed_query(query)
pairs_by_text = vector_store.similarity_search_with_score(query, k=2)
docs_by_vector = vector_store.similarity_search_by_vector(query_vector, k=2)

print("embedding dimensions:", len(query_vector))
print(
    "text API distances:",
    [round(float(distance), 6) for _, distance in pairs_by_text],
)
print(
    "same top source:",
    pairs_by_text[0][0].metadata["source"]
    == docs_by_vector[0].metadata["source"],
)


## 5. MMR 与阈值分别解决什么

- `similarity`：直接取最相关的 `k` 条，适合事实问答基线。
- `mmr`：从更大的 `fetch_k` 候选池中平衡相关性与多样性，适合“概括多个方面”；它不保证精确事实问答更好。`fetch_k` 必须明显大于 `k` 才有意义。
- `similarity_score_threshold`：过滤低相关结果，适合无答案保护；阈值必须用当前 embedding + vector store 的评测分布校准。

`score_threshold` 只属于 `similarity_score_threshold`，不要放进 MMR 参数。


In [ ]:
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 20, "lambda_mult": 0.7},
)
mmr_docs = mmr_retriever.invoke(query)
for rank, document in enumerate(mmr_docs, start=1):
    print(f"#{rank} {Path(document.metadata['source']).name}")
    print(compact(document.page_content), "\n")

# 0.35 只是演示值；生产阈值应通过有答案/无答案验证集校准。
threshold_retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 4, "score_threshold": 0.35},
)


## 6. 当前默认：metadata route + BM25 + dense + RRF

这批数据的查询通常同时包含工程名和任务名。当前 hybrid retriever 先根据 `title + source_name` 识别明确的工程范围，再融合两路排名：

- dense：覆盖口语改写、近义表达；
- 中文 2/3 字符 n-gram BM25：覆盖工程名、任务名、编号、日期等精确词；
- weighted RRF：融合**排名**而不是直接相加两种不可比的原始分数。

`dense_weight=0.40` 是当前 8 条小评测集上的基线参数，不是通用最优值。


In [ ]:
hybrid_hits = retrieve_hybrid(
    vector_store,
    query,
    k=4,
    fetch_k=20,
    dense_weight=0.40,
)
show_hits(hybrid_hits)


## 7. 用固定问题集比较，不凭单次观感调参

当前 8 条问题同时标注了预期来源和答案关键词。它足以做防回归与策略对比，但样本仍很小，不能被解释成泛化结论。下一轮应补充：易混淆负例、问题中不出现工程名、跨 chunk 问题，以及真正无答案的问题。


In [ ]:
eval_path = (
    REPO_ROOT
    / "knowledge"
    / "project_progress"
    / "evals"
    / "retrieval_v4.jsonl"
)
eval_cases = load_eval_cases(eval_path)

dense_report = evaluate_retriever(
    vector_store,
    eval_cases,
    k=4,
    strategy="dense",
)
hybrid_report = evaluate_retriever(
    vector_store,
    eval_cases,
    k=4,
    strategy="hybrid",
    fetch_k=20,
    dense_weight=0.40,
)

for report in (dense_report, hybrid_report):
    print(
        report["strategy"],
        f"source_hit@4={report['source_hit_at_k']:.3f}",
        f"term_hit@4={report['term_hit_at_k']:.3f}",
    )


In [ ]:
# 失败行比汇总分数更重要：它们直接告诉我们该改数据、chunk、路由还是排序。
for row in hybrid_report["rows"]:
    passed = row["source_hit"] and row["term_hit"]
    status = "OK" if passed else "MISS"
    print(f"[{status}] {row['id']}: {row['query']}")
    if not passed:
        print("  top sources:", row["top_sources"][:4])


## 8. Retriever Tool 是接入 Agent 的适配层

`create_retriever_tool` 的价值是把已经验证过的 retriever 暴露给 Agent，让模型决定何时调用。它不会替换 chunk、embedding、BM25、路由或评测，也不会让一个召回不准的 retriever 自动变准。

因此这里只验证工具可以构造，不启动 LLM 或 LangGraph。


In [ ]:
from langchain_core.tools import create_retriever_tool

hybrid_retriever = build_hybrid_retriever(
    vector_store,
    k=4,
    fetch_k=20,
    dense_weight=0.40,
)
retriever_tool = create_retriever_tool(
    hybrid_retriever,
    name="search_project_progress",
    description=(
        "检索输变电工程进度计划，返回相关任务、开始日期、完成日期和来源。"
        "当问题涉及具体工程、任务名称、工期或日期时使用。"
    ),
    response_format="content_and_artifact",
)
print(retriever_tool.name)
print(retriever_tool.description)


## 9. 后续改进顺序

建议按下面的顺序继续，而不是一次堆上所有 LangChain retriever：

1. **扩评测集**：增加无工程名、易混淆、跨 chunk、无答案问题；报告 `hit@k`、首个正确结果排名和失败类型。
2. **修数据与 chunk**：如果正确事实被切断，优先做任务记录级 chunk 或 parent-child retrieval，而不是换 embedding。
3. **校准 hybrid**：在验证集扫描 `dense_weight`、`fetch_k` 和最终 `k`，不要根据一个问题手调。
4. **再加 reranker**：召回池里已有正确答案、但顺序靠后时，再对 top-20 做 cross-encoder rerank。
5. **最后接 query rewrite / Agent**：只在用户问题确实含糊、需要多步或多数据源时引入；保留固定 2-step RAG 作为低延迟基线。

这套顺序也适用于后面的图像 caption：caption 最终仍进入同一 `Document + metadata + retrieval evaluation` 契约。
